# 03 - Silver Layer: Data Cleansing & Validation
Apply validation rules as expectations, drop bad rows, standardize schemas.

**Medallion Architecture - Silver**: Cleansed, validated, standardized data.

In [0]:
%run ./00_setup_config

# 00 - Setup & Configuration
This notebook sets up the database, defines paths, and installs required libraries for the Food Inspection project.

All file paths are parameterized using widgets — no hardcoded values.

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Widget Parameters

Volume Path      : /Volumes/workspace/food_inspection/raw_data
Raw Chicago Path : /Volumes/workspace/food_inspection/raw_data/Food_Inspections_20260411.csv
Raw Dallas Path  : /Volumes/workspace/food_inspection/raw_data/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv
Database Name    : food_inspection


## Create Database

Database 'food_inspection' is ready.


## Verify Raw Data Files Exist

Raw data files in Volume:
  Food_Inspections_20260411.csv (330.70 MB)
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv (192.79 MB)


Configuration ready. All path variables are available via %run.


In [0]:
spark.sql(f"USE {DATABASE_NAME}")

DataFrame[]

In [0]:
from pyspark.sql.functions import (
    col, trim, upper, lower, when, lit, regexp_replace, regexp_extract,
    to_date, split, explode, posexplode, array, struct, expr,
    monotonically_increasing_id, concat_ws, coalesce, length,
    count, sum as spark_sum, row_number, current_timestamp, md5, sha2
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType

In [0]:
def add_silver_lineage(df, key_columns):
    """Add Silver layer lineage columns: updated_at and durability_key (hash of business key columns)."""
    return (
        df
        .withColumn("updated_at", current_timestamp())
        .withColumn("durability_key", sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("NULL")) for c in key_columns]), 256))
    )

## 1. Load Bronze Tables

In [0]:
df_chicago_bronze = spark.table(f"{DATABASE_NAME}.bronze_chicago_inspections")
df_dallas_bronze = spark.table(f"{DATABASE_NAME}.bronze_dallas_inspections")

print(f"Chicago Bronze count: {df_chicago_bronze.count()}")
print(f"Dallas Bronze count: {df_dallas_bronze.count()}")

Chicago Bronze count: 308431
Dallas Bronze count: 78984


---
## 2. Chicago Silver Cleansing

### 2.1 Basic Cleansing & Standardization

In [0]:
df_chicago_clean = (
    df_chicago_bronze
    # Trim whitespace and remove escaped/stray quotes from name columns
    .withColumn("DBA_Name", trim(regexp_replace(col("DBA_Name"), '"', '')))
    .withColumn("AKA_Name", trim(regexp_replace(col("AKA_Name"), '"', '')))
    .withColumn("Address", trim(col("Address")))
    .withColumn("City", trim(upper(col("City"))))
    .withColumn("State", trim(upper(col("State"))))
    .withColumn("Results", trim(col("Results")))
    .withColumn("Inspection_Type", trim(col("Inspection_Type")))
    .withColumn("Facility_Type", trim(col("Facility_Type")))
    .withColumn("Risk", trim(col("Risk")))
    # Cast Zip to string and pad to 5 digits
    .withColumn("Zip", regexp_replace(col("Zip").cast("string"), "\\.0$", ""))
    .withColumn("Zip", when(length(col("Zip")) == 4, concat_ws("", lit("0"), col("Zip"))).otherwise(col("Zip")))
    # Parse Inspection Date
    .withColumn("Inspection_Date", to_date(col("Inspection_Date"), "MM/dd/yyyy"))
    # Cast Latitude/Longitude to double (use try_cast to handle bad data like ' RODENTS' → null)
    .withColumn("Latitude", expr("try_cast(Latitude as double)"))
    .withColumn("Longitude", expr("try_cast(Longitude as double)"))
    # Add source city
    .withColumn("source_city", lit("Chicago"))
)

### 2.2 Derive Inspection Score from Results (Chicago)

In [0]:
df_chicago_clean = df_chicago_clean.withColumn(
    "Inspection_Score",
    when(col("Results") == "Pass", 90)
    .when(col("Results") == "Pass w/ Conditions", 80)
    .when(col("Results") == "Fail", 70)
    .when(col("Results") == "No Entry", 0)
    .otherwise(lit(None).cast(IntegerType()))
)

### 2.3 Apply Validation Rules (Expectations) - Chicago

In [0]:
chicago_before = df_chicago_clean.count()

# --- Tag each row with the FIRST validation rule it fails ---
df_chicago_tagged = (
    df_chicago_clean
    .withColumn("_quarantine_reason",
        when(col("DBA_Name").isNull() | (col("DBA_Name") == ""), lit("Rule 1: DBA_Name is null/empty"))
        .when(col("Inspection_Date").isNull(), lit("Rule 2: Inspection_Date is null"))
        .when(col("Inspection_Type").isNull() | (col("Inspection_Type") == ""), lit("Rule 3: Inspection_Type is null/empty"))
        .when(~col("Zip").rlike("^\\d{5}$") | col("Zip").isNull(), lit("Rule 4: Zip is null or invalid format"))
        .when(col("Results").isNull() | (col("Results") == ""), lit("Rule 5: Results is null/empty"))
        .when(col("Violations").isNull() | (col("Violations") == ""), lit("Rule 6: No violations recorded"))
        .when(
            (upper(col("Results")) == "PASS") & col("Violations").rlike("(?i)(urgent|critical)"),
            lit("Rule 7: PASS with Urgent/Critical violation")
        )
        .otherwise(lit(None).cast("string"))
    )
)

# --- Split into clean and quarantine ---
df_chicago_clean = df_chicago_tagged.filter(col("_quarantine_reason").isNull()).drop("_quarantine_reason")
df_chicago_quarantine = df_chicago_tagged.filter(col("_quarantine_reason").isNotNull())

# --- Persist quarantine table ---
(
    df_chicago_quarantine
    .withColumn("quarantined_at", current_timestamp())
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.quarantine_chicago_inspections")
)

chicago_after = df_chicago_clean.count()
quarantine_count = df_chicago_quarantine.count()

print(f"Chicago: {chicago_before} -> {chicago_after} clean rows")
print(f"Chicago quarantined: {quarantine_count} rows")
print(f"\nQuarantine breakdown:")
display(
    df_chicago_quarantine
    .groupBy("_quarantine_reason")
    .count()
    .orderBy(col("count").desc())
)

Chicago: 308431 -> 221877 clean rows
Chicago quarantined: 86554 rows

Quarantine breakdown:


_quarantine_reason,count
Rule 6: No violations recorded,86472
Rule 4: Zip is null or invalid format,42
Rule 7: PASS with Urgent/Critical violation,39
Rule 3: Inspection_Type is null/empty,1


### 2.4 Parse Chicago Violations into Rows
Chicago violations are pipe-delimited. Each violation has format: `CODE. DESCRIPTION - Comments: TEXT`

In [0]:
# Split violations by pipe delimiter and explode into rows
df_chicago_violations = (
    df_chicago_clean
    .withColumn("violation_array", split(col("Violations"), "\\|"))
    .withColumn("violation_raw", explode(col("violation_array")))
    .withColumn("violation_raw", trim(col("violation_raw")))
    .filter(col("violation_raw") != "")
)

# Extract violation code and description from the raw text
# Format: "CODE. DESCRIPTION - Comments: COMMENT_TEXT"
df_chicago_violations = (
    df_chicago_violations
    .withColumn("violation_code", regexp_extract(col("violation_raw"), r"^(\d+)\.", 1))
    .withColumn("violation_description",
        trim(regexp_extract(col("violation_raw"), r"^\d+\.\s*(.+?)(?:\s*-\s*Comments:.*)?$", 1))
    )
    .withColumn("violation_comments",
        trim(regexp_extract(col("violation_raw"), r"-\s*Comments:\s*(.*)", 1))
    )
    .withColumn("violation_points", lit(None).cast(IntegerType()))  # Chicago doesn't have points per violation
)

# Deduplicate violations per inspection (use code + description as key to avoid
# collapsing different violations when code extraction returns blank)
df_chicago_violations = df_chicago_violations.dropDuplicates(["Inspection_ID", "violation_code", "violation_description"])

In [0]:
display(
    df_chicago_violations
    .select("Inspection_ID", "violation_code", "violation_description", "violation_comments")
    .limit(20)
)

Inspection_ID,violation_code,violation_description,violation_comments
2633604,36,THERMOMETERS PROVIDED & ACCURATE,"OBSERVED NO THERMOMETER IN 2 DOOR ""LOW BOY"" COOLER LOCATED IN FRONT PREP AREA. INSTRUCTED TO PROVIDE."
2633579,22,PROPER COLD HOLDING TEMPERATURES,"OBSERVED THE FOLLOWING COLD TCS FOODS FOUND AT IMPROPER TEMPERATURES DURING TIME OF INSPECTION. NOTED APPROXIMATLEY 50 LBS COMMERCIALLY PACKAGED (BOLOGNA, SALAMI, HOT DOGS, POLISH SAUSAGES, DELI MEAT, APPROX. 85 LBS VARIETY FRESH CHEESES, 70 LBS VARIETY PACKAGED SOUR CREAM, ALL RANGING BETWEEN 51F-57F. NOTED APPROXIMATELY 8 LBS CHORIZO (55F), 6 LBS QUESADILLA FILLING MIX (53F). ALL SAID TCS FOODS STORED INSIDE DISPLAY COOLING UNITS LOCATED IN GROCERY AREAS AND IN UPRIGHT COOLERS STORED IN THE PREP AREA OF THE RESTAURANT. INSTRUCTED TO HOLD ALL COLD FOOD ITEMS @41FR AND BELOW AND MAINTAIN. ALL SAID ITEMS WERE VOLUNTARILY DISCARED BY PIC. PRIORITY VIOLATION. #7-38-005. CITATION ISSUED. TOTAL COST OF DISCARDED ITEMS $500."
2633534,6,"PROPER EATING, TASTING, DRINKING, OR TOBACCO USE",NOTED EMPLOYEES DRINKING AND PREPARING FOOD AT THE KITCHEN PREP AREA. INSTRUCTED MANAGEMENT TO PROVIDE A DESIGNATED AREA FOR EMPLOYEES TO DRINK AND EAT AND NOT AT THE PREP AREAS.
2633481,37,FOOD PROPERLY LABELED; ORIGINAL CONTAINER,"OBSERVED PACKAGED DESSERTS NOT PROPERLY LABELED IN THE PREP COOLER.INSTRUCTED TO LABEL ALL DESSERTS WITH COMMON NAME, NAME OF MANUFACTURER AND ADDRESS, INGREDIENTS, NET CONTENTS, ETC."
2633411,3,"MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE, RESPONSIBILITIES AND REPORTING","OBSERVED NO EMPLOYEE HEALTH POLICY ON-SITE AT THE TIME OF INSPECTION. INSTRUCTED TO OBTAIN AND MAINTAIN. PRIORITY FOUNDATION VIOLATION 7-38-010, NO CITATION ISSUED"
2633342,58,ALLERGEN TRAINING AS REQUIRED,FOUND ONE FOOD ALLERGEN TRAINING CERTIFICATE HAS EXPIRED. INSTRUCTED MUST HAVE UPDATED/VALID FOOD ALLERGEN TRAINING CERTIFICATE.
2633332,51,PLUMBING INSTALLED; PROPER BACKFLOW DEVICES,5-205.15 OBSERVED DRAINPIPE LEAKING UNDER HAND WASHING SINK IN REAR WAREWASHING AREA. INSTRUCTED TO REPAIR LEAKING DRAINPIPE.
2633231,16,FOOD-CONTACT SURFACES: CLEANED & SANITIZED,"OBSERVED THE ICE BIN INTERIOR IN NEED OF CLEANING. INSTRUCTED TO REMOVE ICE, CLEAN."
2633179,53,"TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED",OBSERVED NO SELF CLOSING DEVICE ON TOILET ROOM DOORS BOTH MEN AND WOMENS. MUST INSTALL.
2633180,48,"WAREWASHING FACILITIES: INSTALLED, MAINTAINED & USED; TEST STRIPS",BSERVED FACILITY 3-COMPARTMENT SINK FOR WARE WASHING NOT MEETING REQUIREMENTS. WATER TEMPERATURE OF THE SINK WAS OBSERVED AT 71.0F. INFORMED THE PERSON IN CHARGE THAT THE PROPER WATER TEMPERATURE OF THE 3-COMPARTMENT SINK SHALL BE MAINTAINED AT A MINIMUM TEMPERATURE OF 110F. PRIORITY VIOLATION #7-38-025.


### 2.5 Write Chicago Silver Tables

In [0]:
# Silver - Chicago inspections (one row per inspection)
df_chicago_silver = df_chicago_clean.drop("Violations")  # violations stored separately
# Add lineage: updated_at + durability_key (hash of business key)
df_chicago_silver = add_silver_lineage(df_chicago_silver, ["Inspection_ID", "DBA_Name", "Inspection_Date"])

(
    df_chicago_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.silver_chicago_inspections")
)
print(f"Silver Chicago inspections: {df_chicago_silver.count()} rows")

Silver Chicago inspections: 221877 rows


In [0]:
# Silver - Chicago violations (one row per violation per inspection)
df_chicago_violations_out = (
    df_chicago_violations
    .select(
        "Inspection_ID", "violation_code", "violation_description",
        "violation_comments", "violation_points", "source_city"
    )
)
df_chicago_violations_out = add_silver_lineage(df_chicago_violations_out, ["Inspection_ID", "violation_code"])

(
    df_chicago_violations_out
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.silver_chicago_violations")
)
print("Silver Chicago violations table created.")

Silver Chicago violations table created.


---
## 3. Dallas Silver Cleansing

### 3.1 Basic Cleansing & Standardization

In [0]:
df_dallas_clean = (
    df_dallas_bronze
    .withColumn("Restaurant_Name", trim(col("Restaurant_Name")))
    .withColumn("Street_Address", trim(col("Street_Address")))
    .withColumn("Inspection_Type", trim(col("Inspection_Type")))
    # Standardize Zip to 5 digits (extract first 5 digits from ZIP+4 like '75228-3007' or '752435219')
    .withColumn("Zip_Code", regexp_replace(col("Zip_Code").cast("string"), "\\.0$", ""))
    .withColumn("Zip_Code", regexp_extract(col("Zip_Code"), r"^(\d{5})", 1))
    .withColumn("Zip_Code", when(col("Zip_Code") == "", lit(None)).otherwise(col("Zip_Code")))
    .withColumn("Zip_Code", when(length(col("Zip_Code")) == 4, concat_ws("", lit("0"), col("Zip_Code"))).otherwise(col("Zip_Code")))
    # Parse Inspection Date
    .withColumn("Inspection_Date", to_date(col("Inspection_Date"), "MM/dd/yyyy"))
    # Cast score to integer (use try_cast for ANSI mode safety)
    .withColumn("Inspection_Score", expr("try_cast(Inspection_Score as int)"))
    # Parse Lat/Long from combined field, then try_cast for safety
    .withColumn("Latitude", regexp_extract(col("Lat_Long_Location"), r"\(([^,]+),", 1))
    .withColumn("Latitude", expr("try_cast(Latitude as double)"))
    .withColumn("Longitude", regexp_extract(col("Lat_Long_Location"), r",\s*([^)]+)\)", 1))
    .withColumn("Longitude", expr("try_cast(Longitude as double)"))
    # Add hardcoded city/state and source
    .withColumn("City", lit("DALLAS"))
    .withColumn("State", lit("TX"))
    .withColumn("source_city", lit("Dallas"))
    # Derive inspection result from score (to match Chicago's Results field)
    .withColumn("Results",
        when(col("Inspection_Score") >= 90, lit("Pass"))
        .when(col("Inspection_Score") >= 80, lit("Pass w/ Conditions"))
        .when(col("Inspection_Score") >= 70, lit("Fail"))
        .when(col("Inspection_Score") < 70, lit("Fail"))
        .otherwise(lit(None).cast("string"))
    )
    # Generate a unique inspection ID for Dallas (it doesn't have one)
    .withColumn("Inspection_ID", monotonically_increasing_id())
)

### 3.2 Apply Validation Rules (Expectations) - Dallas

In [0]:
dallas_before = df_dallas_clean.count()

# Count violations per row (needed for Rules 6 and 7)
violation_desc_cols = [c for c in df_dallas_clean.columns if c.startswith("Violation_Description")]
df_dallas_clean = df_dallas_clean.withColumn(
    "violation_count",
    sum([when(col(c).isNotNull() & (col(c) != ""), lit(1)).otherwise(lit(0)) for c in violation_desc_cols])
)

# --- Tag each row with the FIRST validation rule it fails ---
df_dallas_tagged = (
    df_dallas_clean
    .withColumn("_quarantine_reason",
        when(col("Restaurant_Name").isNull() | (col("Restaurant_Name") == ""), lit("Rule 1: Restaurant_Name is null/empty"))
        .when(col("Inspection_Date").isNull(), lit("Rule 2: Inspection_Date is null"))
        .when(col("Inspection_Type").isNull() | (col("Inspection_Type") == ""), lit("Rule 3: Inspection_Type is null/empty"))
        .when(~col("Zip_Code").rlike("^\\d{5}$") | col("Zip_Code").isNull(), lit("Rule 4: Zip_Code is null or invalid format"))
        .when(
            col("Inspection_Score").isNotNull() & ((col("Inspection_Score") < 0) | (col("Inspection_Score") > 100)),
            lit("Rule 5: Inspection_Score out of range (0-100)")
        )
        .when(col("violation_count") < 1, lit("Rule 6: No violations recorded"))
        .when(
            (col("Inspection_Score") >= 90) & (col("violation_count") > 3),
            lit("Rule 7: Score >= 90 with more than 3 violations")
        )
        .otherwise(lit(None).cast("string"))
    )
)

# --- Split into clean and quarantine ---
df_dallas_clean = df_dallas_tagged.filter(col("_quarantine_reason").isNull()).drop("_quarantine_reason")
df_dallas_quarantine = df_dallas_tagged.filter(col("_quarantine_reason").isNotNull())

# --- Persist quarantine table ---
(
    df_dallas_quarantine
    .withColumn("quarantined_at", current_timestamp())
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.quarantine_dallas_inspections")
)

dallas_after = df_dallas_clean.count()
quarantine_count = df_dallas_quarantine.count()

print(f"Dallas: {dallas_before} -> {dallas_after} clean rows")
print(f"Dallas quarantined: {quarantine_count} rows")
print(f"\nQuarantine breakdown:")
display(
    df_dallas_quarantine
    .groupBy("_quarantine_reason")
    .count()
    .orderBy(col("count").desc())
)

Dallas: 78984 -> 54090 clean rows
Dallas quarantined: 24894 rows

Quarantine breakdown:


_quarantine_reason,count
Rule 7: Score >= 90 with more than 3 violations,18298
Rule 6: No violations recorded,6579
Rule 1: Restaurant_Name is null/empty,11
Rule 5: Inspection_Score out of range (0-100),6


### 3.3 Unpivot Dallas Violations (Wide to Long)
Convert 25 violation column groups into individual rows.

In [0]:
# Build arrays of structs for each violation slot (1-25)
# Column names after sanitization: Violation_Description_1, Violation_Points_1, etc.
violation_structs = []
for i in range(1, 26):
    desc_col = f"Violation_Description_{i}"
    pts_col = f"Violation_Points_{i}"
    detail_col = f"Violation_Detail_{i}"
    memo_col = f"Violation_Memo_{i}"

    # Check if columns exist
    if desc_col in df_dallas_clean.columns:
        violation_structs.append(
            struct(
                lit(str(i)).alias("violation_num"),
                col(desc_col).alias("violation_description"),
                col(pts_col).cast(IntegerType()).alias("violation_points"),
                col(detail_col).alias("violation_detail"),
                col(memo_col).alias("violation_comments")
            )
        )

# Create array of violation structs and explode
df_dallas_violations = (
    df_dallas_clean
    .withColumn("violations_array", array(*violation_structs))
    .withColumn("violation", explode(col("violations_array")))
    # Only keep non-null violations
    .filter(col("violation.violation_description").isNotNull() & (col("violation.violation_description") != ""))
    .select(
        "Inspection_ID",
        col("violation.violation_num").alias("violation_num"),
        col("violation.violation_description").alias("violation_description_raw"),
        col("violation.violation_points").alias("violation_points"),
        col("violation.violation_detail").alias("violation_detail"),
        col("violation.violation_comments").alias("violation_comments"),
        "source_city"
    )
)

# Extract violation code from description (format: "*CODE Description text")
df_dallas_violations = (
    df_dallas_violations
    .withColumn("violation_code",
        regexp_extract(col("violation_description_raw"), r"\*?(\d+)\s", 1)
    )
    .withColumn("violation_description",
        trim(regexp_extract(col("violation_description_raw"), r"\*?\d+\s+(.*)", 1))
    )
)

# Deduplicate violations per inspection (use code + description as key to avoid
# collapsing different violations when code extraction returns blank)
df_dallas_violations = df_dallas_violations.dropDuplicates(["Inspection_ID", "violation_code", "violation_description"])

In [0]:
display(
    df_dallas_violations
    .select("Inspection_ID", "violation_code", "violation_description", "violation_points", "violation_comments")
    .limit(20)
)

Inspection_ID,violation_code,violation_description,violation_points,violation_comments
0,21,RFSM - Not On Site,2,null
2,12,A food employee shall comply with an exclusion or RESTRICTION,3,missing employee health policies
3,02,Cold Hold (41øF/45øF or below),3,food in the reach in is at 53 degrees (beef and pork at 53 degrees)
4,21,RFSM - Not On Site,2,null
5,10,Clean Sight and Touch,3,"provide cleaning for the prep, coolers"
9,32,Nonfood surfaces-design to be cleaned easily,2,Cardboard
17,02,Cold Hold (41øF/45øF or below),3,"EGG 47'F, CHICKEN SALAD 47'F, PLEASE TURN DOWN YOUR TEMPERATURE SETTING. OBSERVED SETTING AT 5 ( SETTING RANCE OF COOLER 0-9)"
32,34,Pest Control,1,provide pest records on the premises
35,43,Heating/ventilation: designed and installed,1,Clean dirty air vent covers
35,10,Clean Sight and Touch,3,"CLEAN & SANITIZE SPIGOTS OF ICED TEA DISPENSER, INTERIOR OF ICE MACHINE OF MOLD"


### 3.4 Write Dallas Silver Tables

In [0]:
# Silver - Dallas inspections (one row per inspection)
# Drop the wide violation columns
violation_cols_to_drop = [c for c in df_dallas_clean.columns if c.startswith("Violation_")]
violation_cols_to_drop += ["Lat_Long_Location", "Inspection_Month", "Inspection_Year", "violation_count"]
# Also drop individual street components (we keep Street_Address)
violation_cols_to_drop += ["Street_Number", "Street_Name", "Street_Direction", "Street_Type", "Street_Unit"]

df_dallas_silver = df_dallas_clean.drop(*violation_cols_to_drop)
# Add lineage: updated_at + durability_key
df_dallas_silver = add_silver_lineage(df_dallas_silver, ["Inspection_ID", "Restaurant_Name", "Inspection_Date"])

(
    df_dallas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.silver_dallas_inspections")
)
print(f"Silver Dallas inspections: {df_dallas_silver.count()} rows")

Silver Dallas inspections: 54090 rows


In [0]:
# Silver - Dallas violations (one row per violation per inspection)
df_dallas_violations_out = (
    df_dallas_violations
    .select(
        "Inspection_ID", "violation_code", "violation_description",
        "violation_comments", "violation_points", "source_city"
    )
)
df_dallas_violations_out = add_silver_lineage(df_dallas_violations_out, ["Inspection_ID", "violation_code"])

(
    df_dallas_violations_out
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.silver_dallas_violations")
)
print("Silver Dallas violations table created.")

Silver Dallas violations table created.


---
## 4. Validation Summary

In [0]:
display(spark.sql(f"""
    SELECT 'Chicago Inspections' AS table_name, COUNT(*) AS row_count FROM {DATABASE_NAME}.silver_chicago_inspections
    UNION ALL
    SELECT 'Chicago Violations', COUNT(*) FROM {DATABASE_NAME}.silver_chicago_violations
    UNION ALL
    SELECT 'Dallas Inspections', COUNT(*) FROM {DATABASE_NAME}.silver_dallas_inspections
    UNION ALL
    SELECT 'Dallas Violations', COUNT(*) FROM {DATABASE_NAME}.silver_dallas_violations
"""))

table_name,row_count
Chicago Inspections,221877
Chicago Violations,939571
Dallas Inspections,54090
Dallas Violations,308579


## 4.1 Verify quarantine tables

In [0]:
display(spark.sql(f"""
    SELECT 'Chicago Quarantine' AS table_name, COUNT(*) AS row_count FROM {DATABASE_NAME}.quarantine_chicago_inspections
    UNION ALL
    SELECT 'Dallas Quarantine', COUNT(*) FROM {DATABASE_NAME}.quarantine_dallas_inspections
"""))

table_name,row_count
Chicago Quarantine,86554
Dallas Quarantine,24894


## 5. Cleansing Steps Documentation

### Chicago Cleansing:
1. Trimmed whitespace from all string columns
2. Standardized City/State to uppercase
3. Padded Zip codes to 5 digits, validated format
4. Parsed Inspection Date to date type
5. Derived Inspection Score from Results (Pass=90, Pass w/ Conditions=80, Fail=70, No Entry=0)
6. Dropped rows: null Restaurant Name, Inspection Date, Type, Zip, Results
7. Dropped rows: invalid zip format
8. Dropped rows: no violations
9. Dropped rows: Result=PASS but violations contain Urgent/Critical
10. Parsed pipe-delimited violations into individual rows
11. Extracted violation code and description from unstructured text
12. Deduplicated violations per inspection

### Dallas Cleansing:
1. Trimmed whitespace from string columns
2. Standardized Zip codes to 5 digits, validated format
3. Parsed Inspection Date to date type
4. Parsed Lat/Long from combined location field
5. Added City="DALLAS", State="TX"
6. Generated unique Inspection IDs
7. Derived Inspection Result from Score (>=90=Pass, 80-89=Pass w/ Conditions, <80=Fail)
8. Dropped rows: null Restaurant Name, Inspection Date, Type, Zip
9. Dropped rows: Inspection Score < 0 or > 100 (profiling revealed negative scores e.g. -26, treated as invalid data entry)
10. Dropped rows: no violations
11. Dropped rows: score >= 90 with > 3 violations
12. Unpivoted 25 wide violation columns into individual rows
13. Extracted violation codes from description text
14. Deduplicated violations per inspection

### Quarantine Strategy:
- Rows failing any validation rule are tagged with the specific rule violated
- Tagged rows are split into clean (Silver) and quarantine DataFrames
- Quarantine rows are persisted as Delta tables (`quarantine_chicago_inspections`, `quarantine_dallas_inspections`)
- Each quarantine row retains `_quarantine_reason` (which rule) and `quarantined_at` (timestamp)
- Only clean rows proceed to Silver tables and downstream Gold layer
- Quarantine tables enable auditing, root cause analysis, and potential reprocessing